In [1]:
import pandas as pd
import ssl
import certifi
from urllib.request import urlopen

# Create an SSL context using certifi
ssl_context = ssl.create_default_context(cafile=certifi.where())

# URL of the CSV
persons_url = 'https://data.cityofnewyork.us/api/views/f55k-p6yu/rows.csv?accessType=download'

# Open the URL with SSL context and pass it to pandas
with urlopen(persons_url, context=ssl_context) as response:
    df_persons = pd.read_csv(response, low_memory=False)

# Show the first few rows
df_persons.head()

,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,...,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX
0,10249006,4229554,10/26/2019,9:43,31aa2bc0-f545-444f-8cdb-f1cb5cf00b89,Occupant,Unspecified,19141108.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,U
1,10255054,4230587,10/25/2019,15:15,4629e500-a73e-48dc-b8fb-53124d124b80,Occupant,Unspecified,19144075.0,33.0,Not Ejected,...,Does Not Apply,"Front passenger, if two or more persons, inclu...",Lap Belt & Harness,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,F
2,10253177,4230550,10/26/2019,17:55,ae48c136-1383-45db-83f4-2a5eecfb7cff,Occupant,Unspecified,19143133.0,55.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,M
3,6650180,3565527,11/21/2016,13:05,2782525,Occupant,Unspecified,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Notified Person,NaN,NaN,NaN
4,10255516,4231168,10/25/2019,11:16,e038e18f-40fb-4471-99cf-345eae36e064,Occupant,Unspecified,19144329.0,7.0,Not Ejected,...,Does Not Apply,Right rear passenger or motorcycle sidecar pas...,Lap Belt,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,F


In [2]:
#Repeating the same process for df_persons
print("Missing values in df_persons:")
missing_persons = pd.DataFrame({
    'Missing Count': df_persons.isnull().sum(),
    'Missing Percentage': df_persons.isnull().mean() * 100
})
print(missing_persons[missing_persons['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False))

Missing values in df_persons:
                       Missing Count  Missing Percentage
CONTRIBUTING_FACTOR_2        5812175           98.231365
CONTRIBUTING_FACTOR_1        5812040           98.229083
PED_ACTION                   5810756           98.207382
PED_LOCATION                 5810655           98.205675
SAFETY_EQUIPMENT             3083600           52.115815
EJECTION                     2874447           48.580927
POSITION_IN_VEHICLE          2873984           48.573102
EMOTIONAL_STATUS             2773710           46.878375
BODILY_INJURY                2773667           46.877648
COMPLAINT                    2773660           46.877530
PERSON_AGE                    665746           11.251750
PERSON_SEX                    652898           11.034606
VEHICLE_ID                    246687            4.169248
PED_ROLE                      194889            3.293812
PERSON_ID                         19            0.000321


In [3]:
# Display data types of each column
print(df_persons.dtypes)

UNIQUE_ID                  int64
COLLISION_ID               int64
CRASH_DATE                   str
CRASH_TIME                   str
PERSON_ID                    str
PERSON_TYPE                  str
PERSON_INJURY                str
VEHICLE_ID               float64
PERSON_AGE               float64
EJECTION                     str
EMOTIONAL_STATUS             str
BODILY_INJURY                str
POSITION_IN_VEHICLE          str
SAFETY_EQUIPMENT             str
PED_LOCATION                 str
PED_ACTION                   str
COMPLAINT                    str
PED_ROLE                     str
CONTRIBUTING_FACTOR_1        str
CONTRIBUTING_FACTOR_2        str
PERSON_SEX                   str
dtype: object


In [4]:
# Drop columns with 80% or more missing values
columns_to_drop_persons = [
    'CONTRIBUTING_FACTOR_2', 'CONTRIBUTING_FACTOR_1',
    'PED_ACTION', 'PED_LOCATION'
]

df_persons = df_persons.drop(columns=columns_to_drop_persons)

# Quick check
df_persons.shape

(5916822, 17)

In [5]:
#Repeating the same process for df_persons
print("Missing values in df_persons:")
missing_persons = pd.DataFrame({
    'Missing Count': df_persons.isnull().sum(),
    'Missing Percentage': df_persons.isnull().mean() * 100
})
print(missing_persons[missing_persons['Missing Count'] > 0].sort_values(by='Missing Percentage', ascending=False))

Missing values in df_persons:
                     Missing Count  Missing Percentage
SAFETY_EQUIPMENT           3083600           52.115815
EJECTION                   2874447           48.580927
POSITION_IN_VEHICLE        2873984           48.573102
EMOTIONAL_STATUS           2773710           46.878375
BODILY_INJURY              2773667           46.877648
COMPLAINT                  2773660           46.877530
PERSON_AGE                  665746           11.251750
PERSON_SEX                  652898           11.034606
VEHICLE_ID                  246687            4.169248
PED_ROLE                    194889            3.293812
PERSON_ID                       19            0.000321


In [6]:
# Categorical columns to impute
categorical_cols = [
    'SAFETY_EQUIPMENT', 'EJECTION', 'POSITION_IN_VEHICLE',
    'EMOTIONAL_STATUS', 'BODILY_INJURY', 'COMPLAINT',
    'PERSON_SEX', 'PED_ROLE'
]

# Fill missing categorical values with 'Missing'
df_persons[categorical_cols] = df_persons[categorical_cols].fillna('Missing')

# Fill missing age with median
df_persons['PERSON_AGE'] = df_persons['PERSON_AGE'].fillna(df_persons['PERSON_AGE'].median())

# VEHICLE_ID: fill missing with -1 (optional, identifier-like)
df_persons['VEHICLE_ID'] = df_persons['VEHICLE_ID'].fillna(-1)

# Drop rows with missing PERSON_ID
df_persons = df_persons.dropna(subset=['PERSON_ID'])

# Quick check for remaining missing values
missing_persons_after = pd.DataFrame({
    'Missing Count': df_persons.isnull().sum(),
    'Missing Percentage': df_persons.isnull().mean() * 100
})
missing_persons_after[missing_persons_after['Missing Count'] > 0]

,Missing Count,Missing Percentage


In [8]:
#the same standardization format applied in the previous cell but now on df_persons
for col in df_persons.select_dtypes(include='object').columns:
    df_persons[col] = df_persons[col].astype(str).str.lower().str.strip()

print("df_persons after string standardization (first 5 rows of relevant columns):")
for col in df_persons.select_dtypes(include='object').columns:
    print(f"Column: {col}")
    print(df_persons[col].head())
    print("\n")

/var/folders/fb/hjp4syds4_b_t4g65_8szs_r0000gn/T/ipykernel_73876/125095499.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_persons.select_dtypes(include='object').columns:


df_persons after string standardization (first 5 rows of relevant columns):
Column: CRASH_DATE
0    10/26/2019
1    10/25/2019
2    10/26/2019
3    11/21/2016
4    10/25/2019
Name: CRASH_DATE, dtype: str


Column: CRASH_TIME
0     9:43
1    15:15
2    17:55
3    13:05
4    11:16
Name: CRASH_TIME, dtype: str


Column: PERSON_ID
0    31aa2bc0-f545-444f-8cdb-f1cb5cf00b89
1    4629e500-a73e-48dc-b8fb-53124d124b80
2    ae48c136-1383-45db-83f4-2a5eecfb7cff
3                                 2782525
4    e038e18f-40fb-4471-99cf-345eae36e064
Name: PERSON_ID, dtype: str


Column: PERSON_TYPE
0    occupant
1    occupant
2    occupant
3    occupant
4    occupant
Name: PERSON_TYPE, dtype: str


Column: PERSON_INJURY
0    unspecified
1    unspecified
2    unspecified
3    unspecified
4    unspecified
Name: PERSON_INJURY, dtype: str


Column: EJECTION
0        missing
1    not ejected
2        missing
3        missing
4    not ejected
Name: EJECTION, dtype: str


Column: EMOTIONAL_STATUS
0           

/var/folders/fb/hjp4syds4_b_t4g65_8szs_r0000gn/T/ipykernel_73876/125095499.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_persons.select_dtypes(include='object').columns:


In [9]:
#removing duplicates in df_persons
initial_rows_persons = df_persons.shape[0]
df_persons.drop_duplicates(inplace=True)
final_rows_persons = df_persons.shape[0]
print(f"df_persons: {initial_rows_persons - final_rows_persons} duplicate rows removed. {final_rows_persons} rows remaining.")

df_persons: 0 duplicate rows removed. 5916803 rows remaining.


In [11]:
# Save cleaned persons dataset
df_persons.to_csv('cleaned_persons.csv', index=False)